    # Lab 04 · Fashion-MNIST: Augmentation & Evaluation
    **AI Lab · University of Technology Bahrain** · ⏱ ~15 min · ⚡ GPU

    **You will learn**
    - A harder drop-in replacement for MNIST (clothes, 10 classes)
- Data augmentation layers and BatchNormalization
- Early stopping, per-class accuracy, confusion matrix

    > **First time on Kaggle?** Make sure your account is phone-verified, then in the right-hand panel open
    > **Settings → Accelerator** (choose *GPU T4 x2* when the lab needs it) and **Settings → Internet → On**.
    > Run cells top to bottom with **Shift + Enter**.

In [ ]:
# ⚙️ Setup: run me first
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

ON_KAGGLE = os.path.exists("/kaggle/working")
OUT_DIR = "/kaggle/working" if ON_KAGGLE else "."
gpus = tf.config.list_physical_devices("GPU")
print("Running on Kaggle:", ON_KAGGLE)
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("GPU:", [g.name for g in gpus] if gpus else "none (CPU). Turn on Settings > Accelerator > GPU if this lab needs it")
keras.utils.set_random_seed(42)

In [ ]:
# 📦 Dataset loader: downloads with Keras, falls back to datasets attached via "+ Add Input"
def load_dataset(name):
    """name: 'mnist', 'fashion_mnist' or 'cifar10' -> (x_train, y_train), (x_test, y_test)"""
    try:
        return getattr(keras.datasets, name).load_data()
    except Exception as err:
        if name == "mnist":  # Kaggle-attached copy of mnist.npz, if any
            hits = glob.glob("/kaggle/input/**/mnist.npz", recursive=True)
            if hits:
                d = np.load(hits[0])
                return (d["x_train"], d["y_train"]), (d["x_test"], d["y_test"])
        raise RuntimeError(
            "Could not download the dataset. On Kaggle: right panel > Settings > Internet > On "
            "(needs a phone-verified account), then run this cell again.") from err

In [ ]:
(x_train, y_train), (x_test, y_test) = load_dataset("fashion_mnist")
CLASSES = ["T-shirt", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Boot"]
x_train = (x_train.astype("float32") / 255.0)[..., None]
x_test = (x_test.astype("float32") / 255.0)[..., None]

fig, axes = plt.subplots(1, 10, figsize=(13, 1.8))
for c, ax in enumerate(axes):
    ax.imshow(x_train[np.where(y_train == c)[0][0], :, :, 0], cmap="gray"); ax.set_title(CLASSES[c], fontsize=9); ax.axis("off")
plt.show()

## 1 · Augmentation
Small random shifts and zooms create *new* training images every epoch, which makes the model more robust.

In [ ]:
augment = keras.Sequential([
    layers.RandomTranslation(0.08, 0.08),
    layers.RandomZoom(0.1),
    layers.RandomFlip("horizontal"),
], name="augment")

sample = x_train[:1]
fig, axes = plt.subplots(1, 8, figsize=(11, 1.6))
for ax in axes:
    ax.imshow(augment(sample, training=True)[0, :, :, 0], cmap="gray"); ax.axis("off")
plt.suptitle("Same image, 8 random augmentations"); plt.show()

In [ ]:
def block(f):
    return [layers.Conv2D(f, 3, padding="same", use_bias=False), layers.BatchNormalization(), layers.ReLU()]

model = keras.Sequential([
    keras.Input(shape=(28, 28, 1)), augment,
    *block(32), *block(32), layers.MaxPooling2D(),
    *block(64), *block(64), layers.MaxPooling2D(),
    layers.GlobalAveragePooling2D(), layers.Dropout(0.3),
    layers.Dense(10, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

stop = keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)
history = model.fit(x_train, y_train, epochs=25, batch_size=128, validation_split=0.1,
                    callbacks=[stop], verbose=2)

In [ ]:
# 📈 Learning curves
def plot_history(h):
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    ax[0].plot(h.history["loss"], label="train"); ax[0].plot(h.history["val_loss"], label="validation")
    ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(h.history["accuracy"], label="train"); ax[1].plot(h.history["val_accuracy"], label="validation")
    ax[1].set_title("Accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
    plt.tight_layout(); plt.show()
plot_history(history)

## 2 · Which classes are hard?

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
pred = model.predict(x_test, verbose=0).argmax(1)
print(classification_report(y_test, pred, target_names=CLASSES, digits=3))
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=CLASSES).plot(
    ax=ax, cmap="Blues", colorbar=False, xticks_rotation=45)
plt.show()

## 🧪 Try it
1. Remove the `augment` layer. Does validation accuracy go up or down? What about the gap between train and validation?
2. Shirt vs T-shirt vs Pullover are often confused. Show 10 misclassified shirts. Would *you* get them right?
3. Is `RandomFlip("horizontal")` a sensible augmentation for clothes? Would it be for MNIST digits?

### 💾 Save your work
Click **Save Version** (top right) → *Quick Save* to keep your edits, or *Save & Run All* to store the outputs too.
Files written to `/kaggle/working` appear under **Output** and can be downloaded.